In [5]:
using PlotlyJS, Printf, Random

f1(x) = 5 - 24x + 17x^2 - (11/3)*x^3 + (1/4)*x^4
f2(x) = (x - 8)^2
a, b = -1.0, 9.0

N = 2000
xs = collect(range(a, b, length=N))
f1v = f1.(xs)
f2v = f2.(xs)

f1_min, f1_max = extrema(f1v)
f2_min, f2_max = extrema(f2v)

norm1(x) = (f1_max - f1(x)) / (f1_max - f1_min)
norm2(x) = (f2_max - f2(x)) / (f2_max - f2_min)

@printf("Идеальная точка: (1, 1)\n")
@printf("f₁ ∈ [%.4f, %.4f],  f₂ ∈ [%.4f, %.4f]\n", f1_min, f1_max, f2_min, f2_max)

ρ(x, w1=0.5) = sqrt(w1 * (1 - norm1(x))^2 + (1 - w1) * (1 - norm2(x))^2)

function simulated_annealing(obj, a, b; T0=10.0, alpha=0.995, max_iter=50000, seed=42)
    Random.seed!(seed)
    x = a + rand() * (b - a)
    fx = obj(x)
    best_x, best_f = x, fx
    T = T0
    for _ in 1:max_iter
        dx = randn() * (b - a) * 0.05 * T / T0
        x_new = clamp(x + dx, a, b)
        f_new = obj(x_new)
        if f_new < fx || rand() < exp((fx - f_new) / T)
            x, fx = x_new, f_new
        end
        if fx < best_f
            best_x, best_f = x, fx
        end
        T *= alpha
    end
    best_x, best_f
end


p1 = plot([
    scatter(x=xs, y=f1v, mode="lines", name="f₁(x) = 5−24x+17x²−(11/3)x³+(1/4)x⁴",
            line=attr(color="royalblue", width=2)),
    scatter(x=xs, y=f2v, mode="lines", name="f₂(x) = (x−8)²",
            line=attr(color="crimson", width=2))
], Layout(title="Целевые функции на компакте [$a, $b]",
    xaxis_title="x", yaxis_title="f(x)",
    template="plotly_white", width=850, height=500, legend=attr(x=0.02, y=0.98)))
display(p1)


n1v = norm1.(xs)
n2v = norm2.(xs)

function pareto_front(fv1, fv2)
    idx = sortperm(fv1)
    pf1, pf2 = Float64[], Float64[]
    best = -Inf
    for i in idx
        if fv2[i] > best
            push!(pf1, fv1[i]); push!(pf2, fv2[i])
            best = fv2[i]
        end
    end
    pf1, pf2
end

pf1, pf2 = pareto_front(n1v, n2v)

p2 = plot([
    scatter(x=n1v, y=n2v, mode="markers", name="Область достижимости",
            marker=attr(color="lightgray", size=2)),
    scatter(x=pf1, y=pf2, mode="lines", name="Фронт Парето",
            line=attr(color="green", width=3)),
    scatter(x=[1], y=[1], mode="markers", name="Идеальная точка (1, 1)",
            marker=attr(color="gold", size=14, symbol="star"))
], Layout(title="Фронт Парето (нормированные критерии)",
    xaxis_title="f₁'", yaxis_title="f₂'",
    template="plotly_white", width=850, height=550))
display(p2)


weights_plot = [0.2, 0.5, 0.8]
colors_w = ["#1f77b4", "#2ca02c", "#d62728"]

traces3 = GenericTrace[]
for (i, w) in enumerate(weights_plot)
    rho_vals = [ρ(x, w) for x in xs]
    push!(traces3, scatter(x=xs, y=rho_vals, mode="lines",
        name=@sprintf("w₁=%.1f", w), line=attr(color=colors_w[i], width=2)))
    xo, _ = simulated_annealing(x -> ρ(x, w), a, b)
    push!(traces3, scatter(x=[xo], y=[ρ(xo, w)], mode="markers",
        marker=attr(color=colors_w[i], size=9), showlegend=false))
end

p3 = plot(traces3, Layout(title="ρ(x) — расстояние до идеальной точки",
    xaxis_title="x", yaxis_title="ρ(x, w)",
    template="plotly_white", width=850, height=500, legend=attr(x=0.02, y=0.98)))
display(p3)


all_w = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9, 1.0]

println("\n" * "=" ^ 60)
@printf("  w₁   |  x*     |  f₁(x*)   |  f₂(x*)  |  ρ\n")
println("-" ^ 60)

viridis = ["#440154","#482878","#3e4989","#31688e","#1f9e89",
           "#35b779","#6ece58","#b5de2b","#fde725"]

traces4 = GenericTrace[
    scatter(x=pf1, y=pf2, mode="lines", name="Фронт Парето",
            line=attr(color="green", width=2.5, dash="dot")),
    scatter(x=[1], y=[1], mode="markers", name="Идеальная точка",
            marker=attr(color="gold", size=14, symbol="star"))
]

for (i, w1) in enumerate(all_w)
    xo, rho_val = simulated_annealing(x -> ρ(x, w1), a, b)
    nx, ny = norm1(xo), norm2(xo)
    @printf("  %.1f   | %6.3f  | %8.4f  | %7.4f  | %.4f\n",
            w1, xo, f1(xo), f2(xo), rho_val)

    push!(traces4, scatter(x=[nx, 1], y=[ny, 1], mode="lines",
        line=attr(color=viridis[i], width=1, dash="dash"), showlegend=false))
    push!(traces4, scatter(x=[nx], y=[ny], mode="markers",
        name=@sprintf("w₁=%.1f, ρ=%.3f", w1, rho_val),
        marker=attr(color=viridis[i], size=9)))
end

p4 = plot(traces4, Layout(title="Решения на фронте Парето (имитация отжига)",
    xaxis_title="f₁'", yaxis_title="f₂'",
    template="plotly_white", width=900, height=600,
    legend=attr(x=1.02, y=1.0, xanchor="left")))
display(p4)

Идеальная точка: (1, 1)
f₁ ∈ [-5.4167, 133.2500],  f₂ ∈ [0.0000, 81.0000]


data: [
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields height, legend, margin, template, title, width, xaxis, and yaxis"

data: [
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y"
]

layout: "layout with fields height, margin, template, title, width, xaxis, and yaxis"

data: [
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, showlegend, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, showlegend, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, showlegend, type, x, and y"
]

layout: "layout with fields height, legend, margin, template, title, width, xaxis, and yaxis"


  w₁   |  x*     |  f₁(x*)   |  f₂(x*)  |  ρ
------------------------------------------------------------
  0.0   |  8.009  |  48.2001  |  0.0001  | 0.0000
  0.1   |  6.665  |   7.9502  |  1.7812  | 0.0369
  0.2   |  6.497  |   6.5343  |  2.2601  | 0.0459
  0.3   |  6.384  |   5.8726  |  2.6129  | 0.0521
  0.5   |  6.249  |   5.3460  |  3.0672  | 0.0611
  0.7   |  6.137  |   5.1003  |  3.4699  | 0.0677
  0.8   |  6.081  |   5.0339  |  3.6833  | 0.0704
  0.9   |  6.049  |   5.0121  |  3.8080  | 0.0729
  1.0   |  2.304  |   2.1469  | 32.4438  | 0.0545


data: [
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y",
  "scatter with fields line, mode, showlegend, type, x, and y",
  "scatter with fields marker, mode, name, type, x, and y"
]

layout: "layout with fields height, legend, margin, template, title, width, xaxis, and yaxis"